In [2]:
%pip install selenium beautifulsoup4

import re
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup


%pip install rdflib pandas -U -q

import pandas as pd # for loading CSV
from rdflib import Graph, Literal, Namespace,URIRef
from rdflib.namespace import RDF, RDFS, XSD, SDO, OWL

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
def scrape_book(url, subcategory, max_attempts=2):
    for attempt in range(max_attempts):
        options = Options()
        options.add_argument("--start-maximized")
        driver = webdriver.Chrome(options=options)

        try:
            print(f"  Attempt {attempt + 1}...")
            driver.get(url)

            # Wait for actual product page
            for _ in range(8):
                time.sleep(1)
                soup = BeautifulSoup(driver.page_source, "html.parser")
                page_text = soup.get_text("\n", strip=True)

                if "ISBN:" in page_text and "Number of pages:" in page_text:
                    break
            else:
                print("  Page did not load correctly.")
                continue

            # Title
            title_parts = [x.strip() for x in driver.title.split(" : ")]
            book_title = title_parts[0] if title_parts else None

            if not book_title:
                title_tag = soup.find("h1")
                book_title = title_tag.get_text(" ", strip=True) if title_tag else None

            # Authors from Blackwell's author links
            authors = []

            for link in soup.select('a[href*="/bookshop/search/author/"]'):
                name = link.get_text(" ", strip=True)
                if not name:
                    continue

                context = link.parent.get_text(" ", strip=True).lower()

                non_author_roles = ["translator", "illustrator", "editor", "photographer", "foreword", "introduction", "artist"]

                if not any(role in context for role in non_author_roles):
                    if name not in authors:
                        authors.append(name)

            author = ", ".join(authors) if authors else None

            # Fallback if no author links were found
            if not author and len(title_parts) >= 2:
                author = title_parts[1]
                author = author.replace("(author)", "")
                author = author.replace(": Signed by the Author", "")
                author = author.rstrip(",").strip()

            # Book details
            isbn_match = re.search(r"ISBN:\s*(\d+)", page_text)
            publisher_match = re.search(r"ISBN:.*?Publisher:\s*([^\n]+)", page_text, re.DOTALL)
            date_match = re.search(r"Pub date:\s*([^\n]+)", page_text)
            pages_match = re.search(r"Number of pages:\s*(\d+)", page_text)

            isbn = isbn_match.group(1) if isbn_match else None
            publisher = publisher_match.group(1).strip() if publisher_match else None
            pub_date = date_match.group(1).strip() if date_match else None
            pages = pages_match.group(1) if pages_match else None

            # Current Blackwell's price
            price_tag = soup.find("li", class_="product-price--current")
            price = price_tag.get_text(strip=True) if price_tag else None

            if price:
                price = price.replace(",", ".").replace("€", "").strip()

            # Date → YYYY-MM-DD
            if pub_date:
                pub_date = pd.to_datetime(pub_date).strftime("%Y-%m-%d")

            # Item URI
            safe_title = re.sub(r"[^a-z0-9]+", "-", book_title.lower()).strip("-")
            item_uri = f"http://example.org/webshop/{safe_title}-{isbn}"

            return {
                "Item Name": book_title,
                "Item URL": url,
                "Item URI": item_uri,
                "Category": "Book",
                "Subcategory": subcategory,
                "Publisher": publisher,

                "Attribute 1": "bookTitle",
                "Value 1": book_title,
                "Schema URI 1": "https://schema.org/name",

                "Attribute 2": "author",
                "Value 2": author,
                "Schema URI 2": "https://schema.org/author",

                "Attribute 3": "price",
                "Value 3": price,
                "Schema URI 3": "https://schema.org/price",

                "Attribute 4": "numberOfPages",
                "Value 4": pages,
                "URI 4": "https://schema.org/numberOfPages",

                "Attribute 5": "datePublished",
                "Value 5": pub_date,
                "URI 5": "https://schema.org/datePublished"
            }

        finally:
            driver.quit()

        time.sleep(2)

    print("FAILED TO LOAD:", url)
    return None


books_to_scrape = [
    # ROMANCE — 10
    ("https://blackwells.co.uk/bookshop/product/The-Deal-by-Elle-Kennedy/9780349447155", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Burn-of-the-Everflame-by-Penn-Cole/9781035432523", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/The-Poison-Daughter-by-Sheila-Masterson/9781911760764", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/The-Girls-by-John-Bowen/9781529970982", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/The-Ballad-of-Falling-Dragons-by-Sarah-A-Parker/9780008838126", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Pink-Ink-by-Avina-St-Graves/9781668218983", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Twisted-Lies-by-Ana-Huang/9780349448411", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Riftborne-by-Bree-Grenwich-Parker-Lennox/9780241818657", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Smoke-by-Ivan-Sergeevich-Turgenev-author-Donald-Rayfield-translator/9798896230441", "Romance"),
    ("https://blackwells.co.uk/bookshop/product/Butcher-and-Blackbird-by-Brynne-Weaver/9780349441566", "Romance"),

    # HORROR — 7
    ("https://blackwells.co.uk/bookshop/product/IT-by-Stephen-King/9781399761550", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/Fawn-by-C-N-Vair/9780857508805", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/Solace-House-by-Will-Maclean/2100000300747", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/We-Have-Always-Lived-in-the-Castle-by-Shirley-Jackson/9780141191454", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/Frankenstein-by-Philip-Pullman-Mary-Wollstonecraft-Shelley/9780198314981", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/THE-WALL-A-horrifying-true-story-of-a-haunting-by-Sarro-Cindy/9781420880373", "Horror"),
    ("https://blackwells.co.uk/bookshop/product/Carmilla-by-Joseph-Sheridan-Le-Fanu/9781782275848", "Horror"),
    # FANTASY — 8
    ("https://blackwells.co.uk/bookshop/product/Fourth-Wing-by-Rebecca-Yarros/9780349437019", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/Iron-Flame-by-Rebecca-Yarros/9780349437057", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/Onyx-Storm-by-Rebecca-Yarros/9780349437064", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/Shatter-Me-by-Tahereh-Mafi/9780008660239", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/Wolf-Siren-by-Beth-OBrien/9780008642013", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/The-Enemys-Daughter-by-Melissa-Poett/9780008774585", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/ReZERO-Volume-1-by-Tappei-Nagatsuki-author-Shinichiro-Otsuka-illustrator/9780316315302", "Fantasy"),
    ("https://blackwells.co.uk/bookshop/product/Chain-of-Thorns-by-Cassandra-Clare/9781406358117", "Fantasy"),

    # MATHEMATICS — 6
    ("https://blackwells.co.uk/bookshop/product/The-Big-Book-of-Real-Analysis-by-Syafiq-Johar/9783031308314", "Mathematics"),
    ("https://blackwells.co.uk/bookshop/product/Introducing-Mathematics-by-Ziauddin-Sardar-author-Jerry-Ravetz-author-Borin-Van-Loon-illustrator/9781848312975", "Mathematics"),
    ("https://blackwells.co.uk/bookshop/product/Mathematics-by-Keith-J-Devlin/9780231116398", "Mathematics"),
    ("https://blackwells.co.uk/bookshop/product/Cambridge-O-Level-Mathematics-Coursebook-by-Dean-Chalmers-Ahmed-Saya/9781009316453", "Mathematics"),
    ("https://blackwells.co.uk/bookshop/product/9780198732822", "Mathematics"),
    ("https://blackwells.co.uk/bookshop/product/9780198754046", "Mathematics"),

    # PHILOSOPHY — 6
    ("https://blackwells.co.uk/bookshop/product/The-Myth-of-Sisyphus-by-Albert-Camus/9780141182001", "Philosophy"),
    ("https://blackwells.co.uk/bookshop/product/Principles-of-Biomedical-Ethics-by-Tom-L-Beauchamp-James-F-Childress/9780190640873", "Philosophy"),
    ("https://blackwells.co.uk/bookshop/product/Teaching-About-Technology-by-Marc-J-de-Vries/9781402052743", "Philosophy"),
    ("https://blackwells.co.uk/bookshop/product/Philosophical-Issues-of-Human-Cyborgization-and-the-Necessity-of-Prolegomena-on-Cyborg-Ethics-by-Ivana-Greguric/9781799892311", "Philosophy"),
    ("https://blackwells.co.uk/bookshop/product/9780198778028", "Philosophy"),
    ("https://blackwells.co.uk/bookshop/product/9780192854087", "Philosophy"),

    # TRAVEL — 6
    ("https://blackwells.co.uk/bookshop/product/The-Discovery-of-France-by-Graham-Robb/9781035039197", "Travel"),
    ("https://blackwells.co.uk/bookshop/product/9781784161194", "Travel"),
    ("https://blackwells.co.uk/bookshop/product/9780099769514", "Travel"),
    ("https://blackwells.co.uk/bookshop/product/9781784161446", "Travel"),
    ("https://blackwells.co.uk/bookshop/product/9780141030586", "Travel"),
    ("https://blackwells.co.uk/bookshop/product/9780618658947", "Travel"),

    # CHILDREN'S BOOKS — 7
    ("https://blackwells.co.uk/bookshop/product/The-Baddies-by-Julia-Donaldson-author-Axel-Scheffler-artist/9780702303517", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/The-Stick-Man-Finger-Puppet-Book-by-Julia-Donaldson-author-Axel-Scheffler-artist/9780702335501", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/9781035004232", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/The-Miraculous-Journey-of-Edward-Tulane-by-Kate-DiCamillo-Bagram-Ibatoulline-illustrator/9781406360660", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/Goodnight-Mister-Tom-by-Michelle-Magorian/9780141354804", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/Lolly-Woe-by-Anna-Perera-author-Martina-Selway-illustrator/9780198448549", "ChildrensBook"),
    ("https://blackwells.co.uk/bookshop/product/Harry-Potter-Box-Set-The-Complete-Collection-Childrens-Hardback-by-J-K-Rowling/9781408856789", "ChildrensBook")
]

print("Total books:", len(books_to_scrape))


rows = []

for url, subcategory in books_to_scrape:
    print("\nScraping:", url)
    row = scrape_book(url, subcategory)
    if row is not None:
        rows.append(row)
    time.sleep(1)

df_scraped = pd.DataFrame(rows)

print(
    f"\nSuccessfully scraped "
    f"{len(df_scraped)} / {len(books_to_scrape)} books.")

df_scraped

df_scraped.to_csv(
    "group_10_ken3140_webshop.csv",
    index=False
)

Total books: 50

Scraping: https://blackwells.co.uk/bookshop/product/The-Deal-by-Elle-Kennedy/9780349447155
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/Burn-of-the-Everflame-by-Penn-Cole/9781035432523
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/The-Poison-Daughter-by-Sheila-Masterson/9781911760764
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/The-Girls-by-John-Bowen/9781529970982
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/The-Ballad-of-Falling-Dragons-by-Sarah-A-Parker/9780008838126
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/Pink-Ink-by-Avina-St-Graves/9781668218983
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/Twisted-Lies-by-Ana-Huang/9780349448411
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/Riftborne-by-Bree-Grenwich-Parker-Lennox/9780241818657
  Attempt 1...

Scraping: https://blackwells.co.uk/bookshop/product/Smoke-by